# nb140 — ChEMBL bulk activity fetch (broader than Papyrus++)

Pull ChEMBL's full activity table (~20M records, ~14k targets, ~2.5M compounds) directly via the chembl_downloader package, which downloads the SQLite dump and lets us run any SQL.

Filter to records with `pchembl_value IS NOT NULL` (well-quantified). Save as parquet.

This becomes the reference matrix for the 'analogy chain' (pillar 3): for any PXR compound, find Tanimoto neighbors and retrieve their multi-assay profile across THOUSANDS of targets/assays — not just the 41 NR/P450 targets in nb125.

Expected output: ~10M activity records as parquet.

In [ ]:
import os, subprocess, sys
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chembl-downloader', 'tqdm'], check=False)
import chembl_downloader
print(f'chembl_downloader loaded')
print(f'Latest ChEMBL: {chembl_downloader.latest()}')

In [ ]:
# Download SQLite (cached after first call)
import time
t0 = time.time()
path = chembl_downloader.download_extract_sqlite(version='34', prefix=['kaggle','working','chembl'])
print(f'SQLite at: {path}  size={os.path.getsize(path)//(1024*1024)} MB  in {time.time()-t0:.0f}s')

In [ ]:
# Pull a HUGE activity slice: every record with pchembl_value, with target+compound info
import sqlite3, pandas as pd
from pathlib import Path

OUT = Path('/kaggle/working/chembl_bulk')
OUT.mkdir(exist_ok=True)

conn = sqlite3.connect(str(path))

# Get a count first
n = conn.execute('SELECT COUNT(*) FROM activities WHERE pchembl_value IS NOT NULL').fetchone()[0]
print(f'Total ChEMBL activities with pchembl_value: {n:,}')

# Stream in chunks because the join is huge
QUERY = '''
SELECT
    act.activity_id,
    act.molregno,
    md.chembl_id AS compound_chembl_id,
    cs.canonical_smiles,
    cs.standard_inchi_key,
    act.assay_id,
    a.chembl_id AS assay_chembl_id,
    a.assay_type,
    a.confidence_score,
    a.tid,
    td.chembl_id AS target_chembl_id,
    td.pref_name AS target_name,
    td.organism,
    cs2.accession AS accession,
    act.standard_type,
    act.standard_relation,
    act.standard_value,
    act.standard_units,
    act.pchembl_value,
    act.activity_comment
FROM activities act
JOIN molecule_dictionary md ON act.molregno = md.molregno
JOIN compound_structures cs ON md.molregno = cs.molregno
JOIN assays a ON act.assay_id = a.assay_id
JOIN target_dictionary td ON a.tid = td.tid
LEFT JOIN target_components tc ON td.tid = tc.tid
LEFT JOIN component_sequences cs2 ON tc.component_id = cs2.component_id
WHERE act.pchembl_value IS NOT NULL
  AND a.confidence_score >= 6
  AND cs.canonical_smiles IS NOT NULL
'''
# Stream by activity_id ranges in chunks
max_id = conn.execute('SELECT MAX(activity_id) FROM activities WHERE pchembl_value IS NOT NULL').fetchone()[0]
print(f'Max activity_id: {max_id:,}')

CHUNK = 500_000
kept_chunks = []
t0 = time.time()
for lo in range(0, max_id + 1, CHUNK):
    hi = lo + CHUNK
    df_chunk = pd.read_sql(QUERY + f' AND act.activity_id >= {lo} AND act.activity_id < {hi}', conn)
    if len(df_chunk) > 0:
        kept_chunks.append(df_chunk)
        print(f'  [{lo}, {hi}): kept {len(df_chunk):,}  total={sum(len(c) for c in kept_chunks):,}  ({time.time()-t0:.0f}s)')

df = pd.concat(kept_chunks, ignore_index=True) if kept_chunks else pd.DataFrame()
print(f'\nFinal: {len(df):,} rows  in {time.time()-t0:.0f}s')
if len(df):
    df.to_parquet(OUT / 'chembl_bulk_activities.parquet', index=False)
    print(f'Saved: chembl_bulk_activities.parquet  ({os.path.getsize(OUT / "chembl_bulk_activities.parquet")//(1024*1024)} MB)')

In [ ]:
# Summary stats
if len(df):
    print(f'Unique compounds: {df["compound_chembl_id"].nunique():,}')
    print(f'Unique assays:    {df["assay_chembl_id"].nunique():,}')
    print(f'Unique targets:   {df["target_chembl_id"].nunique():,}')
    print(f'Assay types:      {df["assay_type"].value_counts().to_dict()}')
    print(f'\nTop standard types:')
    print(df['standard_type'].value_counts().head(20))
    print(f'\nTop targets (by record count):')
    print(df.groupby(['target_chembl_id','target_name']).size().sort_values(ascending=False).head(20))
    # Save target summary
    tgt_summary = df.groupby(['target_chembl_id','target_name','accession']).agg(
        n_records=('pchembl_value','count'),
        n_compounds=('compound_chembl_id','nunique'),
        mean_pchembl=('pchembl_value','mean'),
        std_pchembl=('pchembl_value','std'),
    ).reset_index().sort_values('n_records', ascending=False)
    tgt_summary.to_csv(OUT / 'target_summary_bulk.csv', index=False)